In [ ]:
# Si no tienes instaladas las librerías, descomenta la siguiente línea:
# !pip install pandas openpyxl

import pandas as pd
import numpy as np

print("Librerías cargadas correctamente.")

def procesar_plan_de_compras(ruta_archivo):
    # 1. Cargar la planilla Excel en un dataframe 'df'
    df = pd.read_excel(ruta_archivo)
    
    # Identificar nombres exactos de las columnas (insensible a mayúsculas/espacios)
    # df.columns retorna una lista de todas las columnas del DataFrame. Usamos next() para encontrar la primera coincidencia.
    col_codigo = next(col for col in df.columns if col.strip().lower() in ['código presupuestario', 'codigo presupuestario'])
    col_monto = next(col for col in df.columns if col.strip().lower() in ['monto de arrastre', 'monto de arraste'])
    
    filas_procesadas = []
    
    # 2. Iterar sobre cada fila del DataFrame
    for idx, fila in df.iterrows():
        valor_codigo = str(fila[col_codigo]) if pd.notna(fila[col_codigo]) else ""
        
        # Verificar si hay guion que separa múltiples códigos
        if '-' in valor_codigo:
            codigos = [c.strip() for c in valor_codigo.split('-') if c.strip()]
        else:
            codigos = [valor_codigo] if valor_codigo else [np.nan]
            
        # 3. Si hay más de un código, duplicar filas
        if len(codigos) > 1:
            for i, codigo in enumerate(codigos):
                nueva_fila = fila.copy()
                nueva_fila[col_codigo] = codigo
                
                # Para las filas duplicadas (a partir de la 2da), eliminar el Monto De Arrastre
                if i > 0:
                    nueva_fila[col_monto] = np.nan
                    
                filas_procesadas.append(nueva_fila)
        else:
            # Si solo hay un código, mantener la fila tal como está
            filas_procesadas.append(fila)
            
    # Reconstruir el DataFrame procesado
    df_resultado = pd.DataFrame(filas_procesadas)
    
    return df_resultado


In [ ]:
# Ejecutar la función con la planilla
archivo_entrada = 'plan_de_compras_2025.xlsx'
df_procesado = procesar_plan_de_compras(archivo_entrada)

print(f"✅ Procesamiento completado.")
print(f"Filas originales: {len(pd.read_excel(archivo_entrada))}")
print(f"Filas resultantes: {len(df_procesado)}")

In [ ]:
# Guardar en un nuevo archivo Excel
archivo_salida = 'plan_de_compras_2025_gemini.xlsx'
df_procesado.to_excel(archivo_salida, index=False)

print(f"💾 Archivo guardado con éxito como: '{archivo_salida}'")